🔄 Script de Retreinamento Seguro — Safra 2025 (Com Checkpoints)

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
import os
import glob
from google.colab import drive

# 1. Montar Drive (Obrigatório)
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

# --- CONFIGURAÇÕES DA SAFRA 2025 ---
pasta_base = '/content/drive/MyDrive/Tese_IA_Jussara'
SAFRA_ALVO = '2025'

# Busca primeiro um TFRecord MASSIVE da safra-alvo e, se não existir,
# usa qualquer TFRecord da safra-alvo. Assim evitamos treinar 2025 com dados
# antigos por engano.
padroes_busca = [
    f'*{SAFRA_ALVO}*MASSIVE*tfrecord*',
    f'*MASSIVE*{SAFRA_ALVO}*tfrecord*',
    f'*{SAFRA_ALVO}*tfrecord*',
]

busca = []
for padrao in padroes_busca:
    busca = glob.glob(os.path.join(pasta_base, padrao))
    if busca:
        busca.sort(key=os.path.getmtime, reverse=True)
        break

if not busca:
    raise FileNotFoundError(
        f'Nenhum TFRecord da safra {SAFRA_ALVO} encontrado em {pasta_base}. '
        'Confira se o arquivo contém 2025 no nome.'
    )

caminho_arquivo = busca[0]
print(f"📂 Lendo dados da safra {SAFRA_ALVO}: {caminho_arquivo}")

# Parâmetros
KERNEL_SIZE = 128
READ_SIZE = 129
BATCH_SIZE = 32
EPOCHS = 40
INPUT_BANDS = ['R_1', 'NIR_1', 'NDVI_1', 'R_2', 'NIR_2', 'NDVI_2']
LABEL_BAND = 'label_chip'

# --- PIPELINE DE DADOS (Rápido) ---
def parse_and_process(example_proto):
    features_dict = {
        band: tf.io.VarLenFeature(tf.float32) for band in INPUT_BANDS + [LABEL_BAND]
    }
    parsed = tf.io.parse_single_example(example_proto, features_dict)

    inputs_list = []
    for band in INPUT_BANDS:
        dense = tf.sparse.to_dense(parsed[band], default_value=0.0)
        img = tf.reshape(dense, [READ_SIZE, READ_SIZE, 1])
        img = tf.image.resize_with_crop_or_pad(img, KERNEL_SIZE, KERNEL_SIZE)
        inputs_list.append(img)

    image_stacked = tf.concat(inputs_list, axis=-1)

    dense_lbl = tf.sparse.to_dense(parsed[LABEL_BAND], default_value=0.0)
    lbl = tf.reshape(dense_lbl, [READ_SIZE, READ_SIZE, 1])
    lbl = tf.image.resize_with_crop_or_pad(lbl, KERNEL_SIZE, KERNEL_SIZE)
    return image_stacked, lbl

# Contagem rápida para não dar erro de tamanho
print("🔢 Verificando tamanho do arquivo...")
raw_dataset = tf.data.TFRecordDataset(caminho_arquivo, compression_type='GZIP')
N_REAL = sum(1 for _ in raw_dataset)
print(f"✅ Total de amostras: {N_REAL}")

N_TRAIN = int(N_REAL * 0.8)
full_dataset = tf.data.TFRecordDataset(caminho_arquivo, compression_type='GZIP').map(parse_and_process)

train_ds = full_dataset.take(N_TRAIN).cache().shuffle(N_TRAIN).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
val_ds = full_dataset.skip(N_TRAIN).cache().batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

# --- MODELO U-NET ---
def build_unet(input_shape):
    inputs = layers.Input(shape=input_shape)

    # Camadas (Encoder)
    c1 = layers.Conv2D(32, (3, 3), activation='relu', padding='same')(inputs); p1 = layers.MaxPooling2D()(c1)
    c2 = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(p1); p2 = layers.MaxPooling2D()(c2)
    c3 = layers.Conv2D(128, (3, 3), activation='relu', padding='same')(p2); p3 = layers.MaxPooling2D()(c3)

    # Bottleneck
    c4 = layers.Conv2D(256, (3, 3), activation='relu', padding='same')(p3)

    # Decoder
    u5 = layers.Conv2DTranspose(128, (2, 2), strides=(2, 2), padding='same')(c4)
    u5 = layers.concatenate([u5, c3])
    c5 = layers.Conv2D(128, (3, 3), activation='relu', padding='same')(u5)

    u6 = layers.Conv2DTranspose(64, (2, 2), strides=(2, 2), padding='same')(c5)
    u6 = layers.concatenate([u6, c2])
    c6 = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(u6)

    u7 = layers.Conv2DTranspose(32, (2, 2), strides=(2, 2), padding='same')(c6)
    u7 = layers.concatenate([u7, c1])
    c7 = layers.Conv2D(32, (3, 3), activation='relu', padding='same')(u7)

    outputs = layers.Conv2D(1, (1, 1), activation='sigmoid')(c7)
    return models.Model(inputs=[inputs], outputs=[outputs])

model = build_unet((KERNEL_SIZE, KERNEL_SIZE, len(INPUT_BANDS)))
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# --- 🛡️ SALVAMENTO AUTOMÁTICO (SEGURANÇA) ---
# Salva um backup no Drive a cada época para a safra-alvo.
checkpoint_path = os.path.join(pasta_base, f'Modelo_Checkpoint_Jussara_{SAFRA_ALVO}.keras')
checkpoint_cb = callbacks.ModelCheckpoint(
    filepath=checkpoint_path,
    save_best_only=False, # Salva sempre o último estado
    verbose=1
)

print(f"🔥 Iniciando retreinamento da safra {SAFRA_ALVO} (salvamento automático no Drive)...")
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=[checkpoint_cb] # <--- Aqui está a segurança
)

# Salvamento Final Definitivo
final_path = os.path.join(pasta_base, f'Modelo_UNet_Jussara_{SAFRA_ALVO}_FINAL_v2.keras')
model.save(final_path)
print(f"✅ SUCESSO! Modelo final da safra {SAFRA_ALVO} salvo em: {final_path}")


Passo 2: O Teste Visual da Safra 2025 (A Prova de Fogo)
Agora vamos carregar o modelo salvo para 2025 e aplicar em imagens TFRecord da mesma safra para verificar se ele desenha os pivôs.

In [ ]:
import tensorflow as tf
import matplotlib.pyplot as plt
import os
import glob
import numpy as np
from google.colab import drive

# 1. Montar Drive
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

print("--- INICIANDO PROVA REAL DA SAFRA 2025 ---")

# 2. Localizar o Modelo Salvo
pasta_base = '/content/drive/MyDrive/Tese_IA_Jussara'
SAFRA_ALVO = '2025'
caminho_modelo = os.path.join(pasta_base, f'Modelo_UNet_Jussara_{SAFRA_ALVO}_FINAL_v2.keras')

if os.path.exists(caminho_modelo):
    print(f"✅ Arquivo do modelo encontrado: {caminho_modelo}")

    # CARREGAR O MODELO (O momento da verdade)
    try:
        model = tf.keras.models.load_model(caminho_modelo)
        print("✅ Modelo carregado na memória com sucesso!")
    except Exception as e:
        raise RuntimeError(f"❌ ERRO ao carregar modelo: {e}") from e
else:
    raise FileNotFoundError(f"❌ ERRO: o arquivo .keras da safra {SAFRA_ALVO} não foi encontrado em {caminho_modelo}.")

# 3. Carregar um pouco de dados da safra-alvo para testar
padroes_busca = [
    f'*{SAFRA_ALVO}*MASSIVE*tfrecord*',
    f'*MASSIVE*{SAFRA_ALVO}*tfrecord*',
    f'*{SAFRA_ALVO}*tfrecord*',
]

busca_dados = []
for padrao in padroes_busca:
    busca_dados = glob.glob(os.path.join(pasta_base, padrao))
    if busca_dados:
        busca_dados.sort(key=os.path.getmtime, reverse=True)
        break

if not busca_dados:
    raise FileNotFoundError(
        f'Nenhum TFRecord da safra {SAFRA_ALVO} encontrado em {pasta_base}. '
        'Confira se o arquivo contém 2025 no nome.'
    )

caminho_dados = busca_dados[0]
print(f"📂 Validando com dados da safra {SAFRA_ALVO}: {caminho_dados}")

KERNEL_SIZE = 128
READ_SIZE = 129
INPUT_BANDS = ['R_1', 'NIR_1', 'NDVI_1', 'R_2', 'NIR_2', 'NDVI_2']
LABEL_BAND = 'label_chip'

def parse_fast(example_proto):
    features_dict = {band: tf.io.VarLenFeature(tf.float32) for band in INPUT_BANDS + [LABEL_BAND]}
    parsed = tf.io.parse_single_example(example_proto, features_dict)
    inputs_list = []
    for band in INPUT_BANDS:
        dense = tf.sparse.to_dense(parsed[band], default_value=0.0)
        img = tf.reshape(dense, [READ_SIZE, READ_SIZE, 1])
        img = tf.image.resize_with_crop_or_pad(img, KERNEL_SIZE, KERNEL_SIZE)
        inputs_list.append(img)
    image_stacked = tf.concat(inputs_list, axis=-1)
    dense_lbl = tf.sparse.to_dense(parsed[LABEL_BAND], default_value=0.0)
    lbl = tf.reshape(dense_lbl, [READ_SIZE, READ_SIZE, 1])
    lbl = tf.image.resize_with_crop_or_pad(lbl, KERNEL_SIZE, KERNEL_SIZE)
    return image_stacked, lbl

# Pega apenas 1 lote de 10 imagens
dataset = tf.data.TFRecordDataset(caminho_dados, compression_type='GZIP')
dataset = dataset.map(parse_fast).batch(10).take(1)

# 4. Gerar Previsões
print("🔮 Gerando previsões com o modelo carregado...")
imgs, labels = next(iter(dataset))
preds = model.predict(imgs)

# 5. Visualizar
plt.figure(figsize=(15, 12))
print("\nLEGENDA: Esquerda=Satélite | Meio=Gabarito | Direita=O que a IA Aprendeu")

for i in range(5): # Mostra 5 exemplos
    # Satélite (NDVI Safra)
    plt.subplot(5, 3, i*3 + 1)
    plt.imshow(imgs[i][:,:,2], cmap='RdYlGn', vmin=0, vmax=0.8)
    plt.axis('off')
    if i==0: plt.title(f'Satélite (NDVI {SAFRA_ALVO})')

    # Gabarito
    plt.subplot(5, 3, i*3 + 2)
    plt.imshow(labels[i][:,:,0], cmap='binary_r')
    plt.axis('off')
    if i==0: plt.title('Gabarito Real')

    # Predição da IA
    plt.subplot(5, 3, i*3 + 3)
    # Vmin/Vmax fixos para ver a confiança real
    plt.imshow(preds[i][:,:,0], cmap='magma', vmin=0, vmax=1)
    plt.axis('off')
    if i==0: plt.title(f'IA (Safra {SAFRA_ALVO})')

plt.tight_layout()
plt.show()
